In [5]:
import importlib
import numpy as np
import torch

from common import local_rlct_estimater
import objective_function.quadratic_function as quadratic_function

importlib.reload(local_rlct_estimater)
importlib.reload(quadratic_function)

LocalRLCTTorchEstimator = local_rlct_estimater.LocalRLCTTorchEstimator
QuadraticTorchModel = quadratic_function.QuadraticTorchModel


In [6]:
torch.manual_seed(0)

# Change only this line to switch to another model implementation.
quad_dim = 10
model = QuadraticTorchModel(dim=quad_dim)

# Dummy input data is enough because the model objective depends only on parameters.
dummy_x = torch.zeros(1, 1, dtype=torch.float64)
w0 = torch.nn.utils.parameters_to_vector(
    [param.detach().clone() for param in model.parameters()]
)

estimator = LocalRLCTTorchEstimator.from_tensors(
    model,
    lambda m, x: m(x).sum(),
    dummy_x,
    w0=w0,
    scale=1.0,
)

# The estimator now uses SGLD updates internally.
# For tensor inputs, set update_batch_size < dataset_size to simulate minibatch SGLD.
result = estimator.estimate(
    betas=[8, 16, 32, 64, 128],
    step_size=5e-4,
    n_steps=4000,
    burn_in=200,
    thinning=10,
    clip_radius=np.sqrt(quad_dim),
    grad_clip=20.0,
    n_chains=4,
    max_beta_step=0.1,
    regression_tail=None,
    use_weighted_regression=True,
    seed=0,
)

print(f"theory_lambda={quad_dim / 2:.3f}")
print(f"estimated_lambda={result.lambda_hat:.6f}")
print(f"betaEf={result.betaEf}")
print(f"betaEf_se={result.betaEf_se}")

result.as_dict()


theory_lambda=5.000
estimated_lambda=5.418103
betaEf=[4.96819981 5.14084527 5.1274219  5.12636481 5.29437411]
betaEf_se=[0.05180159 0.05844667 0.05962728 0.05734833 0.06070181]


{'lambda_hat': 5.418103068509504,
 'betas': array([  8.,  16.,  32.,  64., 128.]),
 'betaEf': array([4.96819981, 5.14084527, 5.1274219 , 5.12636481, 5.29437411]),
 'mean_f': array([0.62102498, 0.32130283, 0.16023193, 0.08009945, 0.0413623 ]),
 'ess_like_counts': array([1520, 1520, 1520, 1520, 1520]),
 'x0': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 'objective_info': {'loss0': 0.0,
  'scale': 1.0,
  'dataset_size': 1,
  'device': 'cpu',
  'dtype': 'torch.float64',
  'num_params': 10,
  'eval_mode': True,
  'sampler': 'sgld',
  'n_chains': 4,
  'base_step_size': 0.0005,
  'max_beta_step': 0.1,
  'update_batch_size': None,
  'eval_max_batches': None,
  'replace_batches': True,
  'regression_tail': None,
  'use_weighted_regression': True,
  'regression_betas': array([  8.,  16.,  32.,  64., 128.])},
 'betaEf_std': array([2.01959788, 2.27867058, 2.32469945, 2.23584946, 2.36659205]),
 'betaEf_se': array([0.05180159, 0.05844667, 0.05962728, 0.05734833, 0.06070181])}